<a href="https://colab.research.google.com/github/jahirr01/understanding-the-concepts-of-ML/blob/main/Day_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Data Cleaning and feature Extraction**

In [1]:
import pandas as pd
import numpy as np

#Load CSV into DataFrame
df = pd.read_csv("iot_device_data.csv")

print("Original Data:")
print(df.head())

Original Data:
  device_id  temperature  humidity  usage_hours  error_count  vibration  \
0      D101         57.0        41           24            4       0.44   
1      D102         32.0        46           22            1       0.93   
2      D103         26.6        45            7            3       0.81   
3      D104         26.3        52           23            6       0.41   
4      D105         54.5        40            6            6       0.58   

  device_status  
0       warning  
1        normal  
2       warning  
3       failure  
4       failure  


In [2]:
#Data Cleaning

# Replace empty values with NaN
df.replace("", np.nan, inplace=True)

# Fill missing values
df.fillna({
    "temperature": df["temperature"].mean(),
    "humidity": df["humidity"].mean(),
    "usage_hours": 0,
    "error_count": 0,
    "vibration": df["vibration"].median()
}, inplace=True)

# Convert data types
numeric_cols = ["temperature", "humidity", "usage_hours", "error_count", "vibration"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Remove duplicates
df.drop_duplicates(inplace=True)

# Handle outliers
df["temperature"] = np.clip(df["temperature"], 20, 80)
df["humidity"] = np.clip(df["humidity"], 30, 100)
df["vibration"] = np.clip(df["vibration"], 0, 2)

# Standardize text
df["device_status"] = df["device_status"].str.lower().str.strip()

df

,device_id,temperature,humidity,usage_hours,error_count,vibration,device_status
0,D101,57.0,41,24,4,0.44,warning
1,D102,32.0,46,22,1,0.93,normal
2,D103,26.6,45,7,3,0.81,warning
3,D104,26.3,52,23,6,0.41,failure
4,D105,54.5,40,6,6,0.58,failure
...,...,...,...,...,...,...,...
95,D196,62.6,83,10,3,1.23,failure
96,D197,32.4,42,8,7,0.96,failure
97,D198,63.4,69,14,3,1.11,failure
98,D199,44.2,65,8,2,1.02,failure


In [3]:

#Feature Engineering

# 1. Average Temperature per device
df["avg_temp"] = df.groupby("device_id")["temperature"].transform("mean")

# 2. Temperature Change (difference from previous reading)
df["temp_change"] = df.groupby("device_id")["temperature"].diff().fillna(0)

# 3. Usage Frequency (usage per hour approximation)
df["usage_frequency"] = df["usage_hours"] / (df["usage_hours"].max())

# 4. Health Score (reuse + improved)
df["health_score"] = (
    100
    - df["temperature"] * 0.3
    - df["error_count"] * 5
    - df["vibration"] * 10
)

# 5. Health Category
df["health_status"] = np.where(
    df["health_score"] >= 85, "excellent",
    np.where(
        df["health_score"] >= 70, "good",
        np.where(df["health_score"] >= 40, "moderate", "critical")
    )
)

# 6. Risk Flag (ML target feature)
df["is_risky"] = np.where(df["device_status"] == "failure", 1, 0)
df

,device_id,temperature,humidity,usage_hours,error_count,vibration,device_status,avg_temp,temp_change,usage_frequency,health_score,health_status,is_risky
0,D101,57.0,41,24,4,0.44,warning,57.0,0.0,1.000000,58.50,moderate,0
1,D102,32.0,46,22,1,0.93,normal,32.0,0.0,0.916667,76.10,good,0
2,D103,26.6,45,7,3,0.81,warning,26.6,0.0,0.291667,68.92,moderate,0
3,D104,26.3,52,23,6,0.41,failure,26.3,0.0,0.958333,58.01,moderate,1
4,D105,54.5,40,6,6,0.58,failure,54.5,0.0,0.250000,47.85,moderate,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,D196,62.6,83,10,3,1.23,failure,62.6,0.0,0.416667,53.92,moderate,1
96,D197,32.4,42,8,7,0.96,failure,32.4,0.0,0.333333,45.68,moderate,1
97,D198,63.4,69,14,3,1.11,failure,63.4,0.0,0.583333,54.88,moderate,1
98,D199,44.2,65,8,2,1.02,failure,44.2,0.0,0.333333,66.54,moderate,1


In [ ]:
#Convert Raw → Structured Dataset

# Select important ML-ready features
structured_df = df[[
    "device_id",
    "temperature",
    "humidity",
    "usage_hours",
    "error_count",
    "vibration",
    "avg_temp",
    "temp_change",
    "usage_frequency",
    "health_score",
    "health_status",
    "is_risky"
]]

print("\n Structured Dataset Sample:")
print(structured_df.head())





 Structured Dataset Sample:
  device_id  temperature  humidity  usage_hours  error_count  vibration  \
0      D101         57.0        41           24            4       0.44   
1      D102         32.0        46           22            1       0.93   
2      D103         26.6        45            7            3       0.81   
3      D104         26.3        52           23            6       0.41   
4      D105         54.5        40            6            6       0.58   

   avg_temp  temp_change  usage_frequency  health_score health_status  \
0      57.0          0.0         1.000000         58.50      moderate   
1      32.0          0.0         0.916667         76.10          good   
2      26.6          0.0         0.291667         68.92      moderate   
3      26.3          0.0         0.958333         58.01      moderate   
4      54.5          0.0         0.250000         47.85      moderate   

   is_risky  
0         0  
1         0  
2         0  
3         1  
4         1

In [ ]:
#Save ML-ready dataset
structured_df.to_csv("ml_ready_iot_data.csv", index=False)

print("\n ML-ready dataset saved as: ml_ready_iot_data.csv")